# Imports - utils

In [1]:
import os
import json
import pandas as pd

In [2]:
if 'google.colab' in str(get_ipython()): # running in colab
    !git clone --config lfs.fetchinclude="*"  https://github.com/sinc-lab/xvalRNAfolding.git
    DATA_PATH = './xvalRNAfolding/'
else:
    DATA_PATH = '/DATA/ncRNA/xval/'

In [3]:
def get_tp(bp_x, bp_ref, strict):
    tp = 0
    for rbp in bp_x:
        cond = rbp in bp_ref
        if not strict:
            cond = cond or [rbp[0], rbp[1] - 1] in bp_ref or [rbp[0], rbp[1] + 1] in bp_ref or [rbp[0] + 1, rbp[1]] in bp_ref or [rbp[0] - 1, rbp[1]] in bp_ref
        if cond:
            tp += 1
    return tp

def bp_metrics(ref_bp, pre_bp, strict=True):
    """F1, recall and precision score from base pairs. Is the same as triangular but less efficient. strict=False takes into account the  neighbors for each base pair as correct"""
    assert all([type(bp) == list for bp in ref_bp]), "ref_bp must be a list of lists"
    assert all([type(bp) == list for bp in pre_bp]), "pre_bp must be a list of lists"

    # corner case when there are no positives
    if len(ref_bp) == 0 and len(pre_bp) == 0:
        return 1.0, 1.0, 1.0

    tp1 = get_tp(pre_bp, ref_bp, strict)
    tp2 = get_tp(ref_bp, pre_bp, strict)

    fn = len(ref_bp) - tp1
    fp = len(pre_bp) - tp1

    tpr = pre = f1 = 0.0
    if tp1 + fn > 0:
        tpr = tp1 / float(tp1 + fn)  # sensitivity (=recall =power)
    if tp1 + fp > 0:
        pre = tp2 / float(tp1 + fp)  # precision (=ppv)
    if tpr + pre > 0:
        f1 = 2 * pre * tpr / (pre + tpr)  # F1 score

    return f1, tpr, pre

def dot2bp(struct):
    bp = []
    if not set(struct).issubset(
        set(["."] + [c for par in MATCHING_BRACKETS for c in par])
    ):
        return False

    for brackets in MATCHING_BRACKETS:
        if brackets[0] in struct:
            bpk = fold2bp(struct, brackets[0], brackets[1])
            if bpk:
                bp = bp + bpk
            else:
                return False
    return list(sorted(bp))

    NT_DICT = {
    "R": ["G", "A"],
    "Y": ["C", "U"],
    "K": ["G", "U"],
    "M": ["A", "C"],
    "S": ["G", "C"],
    "W": ["A", "U"],
    "B": ["G", "U", "C"],
    "D": ["G", "A", "U"],
    "H": ["A", "C", "U"],
    "V": ["G", "C", "A"],
    "N": ["G", "A", "C", "U"],
}

VOCABULARY = ["A", "C", "G", "U"]

MATCHING_BRACKETS = [
    ["(", ")"],
    ["[", "]"],
    ["{", "}"],
    ["<", ">"],
    ["A", "a"],
    ["B", "a"],
]

def fold2bp(struc, xop="(", xcl=")"):
    """Get base pairs from one page folding (using only one type of brackets).
    BP are 1-indexed"""
    openxs = []
    bps = []
    if struc.count(xop) != struc.count(xcl):
        return False
    for i, x in enumerate(struc):
        if x == xop:
            openxs.append(i)
        elif x == xcl:
            if len(openxs) > 0:
                bps.append([openxs.pop() + 1, i + 1])
            else:
                return False
    return bps

# Data

In [4]:
dataset = pd.read_csv(f"{DATA_PATH}data/archiveII.csv", index_col="id")

In [5]:
classical_methods = ["RNAfold", "RNAstructure", "LinearFoldV", "LinearPartitionV", "ProbKnot", "IPKnot"]
trained_methods = ["MxFold2", "REDfold", "UFold", "sincFold"]

# Classical methods

In [6]:
# load predictions and compute f1 scores
classical_summary = []
for method in classical_methods:
    print(method, end=" ")
    pred = pd.read_csv(DATA_PATH+f"results/{method}.csv", index_col="id")
    pred["ref"] = dataset.loc[pred.index, "base_pairs"]
    pred[["f1", "rec", "pre"]] = pred.apply(lambda x: bp_metrics(json.loads(x["ref"]), json.loads(x["base_pairs"])), axis=1, result_type="expand")
    pred["method"] = method
    pred = pred[["method", "f1", "rec", "pre"]]
    classical_summary.append(pred)
classical_summary = pd.concat(classical_summary)

RNAfold RNAstructure LinearFoldV LinearPartitionV ProbKnot IPKnot 

In [7]:
classical_summary.to_pickle(DATA_PATH + "results/classical_summary.pkl")

# Random k-folds

In [8]:
partition_name = "random_kfolds"
splits_kfold = pd.read_csv(f"{DATA_PATH}data/{partition_name}_split.csv", index_col="id")

In [9]:
folds = sorted(splits_kfold["fold_number"].unique())
summary_kfold = []
for fold in folds:
    partition_set = splits_kfold[(splits_kfold["fold_number"] == fold) & (splits_kfold["partition"] == "test")]
    print(fold, end=" ")
    for method in trained_methods:
        f = DATA_PATH+f"results/{partition_name}/{method}/{fold}/pred.csv"
        if not os.path.exists(f):
            print(f"\n{method, fold} not found, skipping\n")
            continue

        pred = pd.read_csv(f, index_col="id")
        pred["ref"] = dataset.loc[partition_set.index, "base_pairs"]
        pred[["f1", "rec", "pre"]] = pred.apply(lambda x: bp_metrics(json.loads(x["ref"]), json.loads(x["base_pairs"])), axis=1, result_type="expand")
        pred["method"] = method
        pred["fold"] = fold
        pred = pred[["method", "fold", "f1", "rec", "pre"]]
        summary_kfold.append(pred)
summary_kfold = pd.concat(summary_kfold)

0 1 2 3 4 

In [10]:
# add classical methods to kfold summary
for method in classical_methods:
    for fold in folds:
        partition_set = splits_kfold[(splits_kfold["fold_number"] == fold) & (splits_kfold["partition"] == "test")]
        res = classical_summary.loc[partition_set.index]
        res = res[res["method"] == method]
        res["fold"] = fold
        summary_kfold = pd.concat([summary_kfold, res])
summary_kfold.method.unique()

array(['MxFold2', 'REDfold', 'UFold', 'sincFold', 'RNAfold',
       'RNAstructure', 'LinearFoldV', 'LinearPartitionV', 'ProbKnot',
       'IPKnot'], dtype=object)

In [11]:
summary_kfold.to_pickle(DATA_PATH + f"results/summary_{partition_name}.pkl")

# Clustering folds

In [12]:
partition_name = "clustering_folds"
splits_cfold = pd.read_csv(f"{DATA_PATH}data/{partition_name}_split.csv", index_col="id")

In [13]:
folds = sorted(splits_cfold["fold_number"].unique())
summary_cfold = []
for fold in folds:
    partition_set = splits_cfold[(splits_cfold["fold_number"] == fold) & (splits_cfold["partition"] == "test")]
    print(fold, end=" ")
    for method in trained_methods:
        f = DATA_PATH+f"results/{partition_name}/{method}/{fold}/pred.csv"
        if not os.path.exists(f):
            print(f"\n{method, fold} not found, skipping\n")
            continue
        print(method, end=" ")

        pred = pd.read_csv(f, index_col="id")
        pred["ref"] = dataset.loc[partition_set.index, "base_pairs"]
        pred[["f1", "rec", "pre"]] = pred.apply(lambda x: bp_metrics(json.loads(x["ref"]), json.loads(x["base_pairs"])), axis=1, result_type="expand")
        pred["method"] = method
        pred["fold"] = fold
        pred = pred[["method", "fold", "f1", "rec", "pre"]]
        summary_cfold.append(pred)
    print()
summary_cfold = pd.concat(summary_cfold)

0 MxFold2 REDfold UFold sincFold 
1 MxFold2 REDfold UFold sincFold 
2 MxFold2 REDfold UFold sincFold 
3 MxFold2 REDfold UFold sincFold 
4 MxFold2 REDfold UFold sincFold 


In [14]:
# add classical methods to the summary
for method in classical_methods:
    for fold in folds:
        partition_set = splits_cfold[(splits_cfold["fold_number"] == fold) & (splits_cfold["partition"] == "test")]
        res = classical_summary.loc[partition_set.index]
        res = res[res["method"] == method]
        res["fold"] = fold
        summary_cfold = pd.concat([summary_cfold, res])
summary_cfold.method.unique()

array(['MxFold2', 'REDfold', 'UFold', 'sincFold', 'RNAfold',
       'RNAstructure', 'LinearFoldV', 'LinearPartitionV', 'ProbKnot',
       'IPKnot'], dtype=object)

In [15]:
summary_cfold.to_pickle(DATA_PATH + f"results/summary_{partition_name}.pkl")

# Family folds

In [16]:
partition_name = "fam_folds"
splits_ffold = pd.read_csv(f"{DATA_PATH}data/{partition_name}_split.csv", index_col="id")

In [17]:
folds = sorted(splits_ffold["fold_number"].unique())
summary_ffold = []
for fold in folds:
    partition_set = splits_ffold[(splits_ffold["fold_number"] == fold) & (splits_ffold["partition"] == "test")]
    print(fold, end=" ")
    for method in trained_methods:
        f = DATA_PATH+f"results/{partition_name}/{method}/{fold}/pred.csv"
        if not os.path.exists(f):
            print(f"\n{method, fold} not found, skipping\n")
            continue
        print(method, end=" ")

        pred = pd.read_csv(f, index_col="id")
        pred["ref"] = dataset.loc[partition_set.index, "base_pairs"]
        pred[["f1", "rec", "pre"]] = pred.apply(lambda x: bp_metrics(json.loads(x["ref"]), json.loads(x["base_pairs"])), axis=1, result_type="expand")
        pred["method"] = method
        pred["fold"] = fold
        pred = pred[["method", "fold", "f1", "rec", "pre"]]
        summary_ffold.append(pred)
summary_ffold = pd.concat(summary_ffold)

0 MxFold2 REDfold UFold sincFold 1 MxFold2 REDfold UFold sincFold 2 MxFold2 REDfold UFold sincFold 3 MxFold2 REDfold UFold sincFold 4 MxFold2 REDfold UFold sincFold 5 MxFold2 REDfold UFold sincFold 6 MxFold2 REDfold UFold sincFold 7 MxFold2 REDfold UFold sincFold 8 MxFold2 REDfold UFold sincFold 

In [18]:
# add classical methods to the summary
for method in classical_methods:
    for fold in folds:
        partition_set = splits_ffold[(splits_ffold["fold_number"] == fold) & (splits_ffold["partition"] == "test")]
        res = classical_summary.loc[partition_set.index]
        res = res[res["method"] == method]
        res["fold"] = fold
        summary_ffold = pd.concat([summary_ffold, res])
summary_ffold.method.unique()

array(['MxFold2', 'REDfold', 'UFold', 'sincFold', 'RNAfold',
       'RNAstructure', 'LinearFoldV', 'LinearPartitionV', 'ProbKnot',
       'IPKnot'], dtype=object)

In [19]:
family_name = {n: splits_ffold[splits_ffold.fold_number==n].iloc[0].fold_name for n in sorted(splits_ffold.fold_number.unique())}
summary_ffold["family"] = summary_ffold.fold.apply(lambda x: family_name[x])

In [20]:
summary_ffold.to_pickle(DATA_PATH + f"results/summary_{partition_name}.pkl")

# Human learned folds

In [21]:
partition_name = "hl_folds"
splits_hlfold = pd.read_csv(f"{DATA_PATH}data/{partition_name}_split.csv", index_col="id")

In [22]:
#folds = sorted(splits["fold_name"].unique())
folds =['hl10','hl20','hl30','hl40','hl50','hl60','hl70','hl80','hl90']
summary_hlfold = []
for fold in folds:
    partition_set = splits_hlfold[(splits_hlfold["fold_name"] == fold) & (splits_hlfold["partition"] == "test")]
    print(fold, end=" ")
    for method in trained_methods:
        f = DATA_PATH+f"results/{partition_name}/{method}/{fold}/pred.csv"
        if not os.path.exists(f):
            print(f"\n{method, fold} not found, skipping\n")
            continue
        print(method, end=" ")

        pred = pd.read_csv(f, index_col="id")
        pred["ref"] = dataset.loc[partition_set.index, "base_pairs"]
        pred[["f1", "rec", "pre"]] = pred.apply(lambda x: bp_metrics(json.loads(x["ref"]), json.loads(x["base_pairs"])), axis=1, result_type="expand")
        pred["method"] = method
        pred["fold"] = fold
        pred = pred[["method", "fold", "f1", "rec", "pre"]]
        summary_hlfold.append(pred)
    print()
summary_hlfold = pd.concat(summary_hlfold)

hl10 MxFold2 REDfold UFold sincFold 
hl20 MxFold2 REDfold UFold sincFold 
hl30 MxFold2 REDfold UFold sincFold 
hl40 MxFold2 REDfold UFold sincFold 
hl50 MxFold2 REDfold UFold sincFold 
hl60 MxFold2 REDfold UFold sincFold 
hl70 MxFold2 REDfold UFold sincFold 
hl80 MxFold2 REDfold UFold sincFold 
hl90 MxFold2 REDfold UFold sincFold 


In [23]:
# add classical methods to the summary
for method in classical_methods:
    for fold in folds:
        partition_set = splits_hlfold[(splits_hlfold["fold_name"] == fold) & (splits_hlfold["partition"] == "test")]
        res = classical_summary.loc[partition_set.index]
        res = res[res["method"] == method]
        res["fold"] = fold
        summary_hlfold = pd.concat([summary_hlfold, res])
summary_hlfold.method.unique()

array(['MxFold2', 'REDfold', 'UFold', 'sincFold', 'RNAfold',
       'RNAstructure', 'LinearFoldV', 'LinearPartitionV', 'ProbKnot',
       'IPKnot'], dtype=object)

In [24]:
summary_hlfold.to_pickle(DATA_PATH + f"results/summary_{partition_name}.pkl")

# Similarity folds

In [28]:
partition_name = "sim_folds"
splits_sfold = []
for f in os.listdir(f"{DATA_PATH}data/{partition_name}/"):
    print(f, end=" ")
    split = pd.read_csv(f"{DATA_PATH}data/{partition_name}/{f}")
    split["similarity"] = f.split("_")[-1].split(".")[0]
    splits_sfold.append(split)
splits_sfold = pd.concat(splits_sfold)

sim90.csv sim50.csv sim80.csv sim60.csv sim30.csv sim40.csv sim70.csv 

In [29]:
similarities = sorted(splits_sfold["similarity"].unique())
similarities = similarities[1:]
folds = splits_sfold["fold_number"].unique()
summary_sfold = []
for similarity in similarities:
    print(similarity, end=" ")
    for fold in folds:
        partition_set = splits_sfold[(splits_sfold["fold_number"] == fold) & (splits_sfold["similarity"] == similarity) & (splits_sfold["partition"] == "test")].set_index('id')

        for method in trained_methods:
            f = DATA_PATH+f"results/{partition_name}/{method}/{similarity}/{fold}/pred.csv"
            if not os.path.exists(f):
                print(f"\n{method, fold} not found, skipping\n")
                continue
            print(method, end=" ")

            pred = pd.read_csv(f, index_col="id")

            pred["ref"] = dataset.loc[partition_set.index, "base_pairs"]
            pred[["f1", "rec", "pre"]] = pred.apply(lambda x: bp_metrics(json.loads(x["ref"]), json.loads(x["base_pairs"])), axis=1, result_type="expand")
            pred["similarity"] = similarity
            pred["method"] = method
            pred["fold"] = fold
            pred = pred[["method", "similarity", "fold", "f1", "rec", "pre"]]
            summary_sfold.append(pred)
summary_sfold = pd.concat(summary_sfold)

sim40 MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold sim50 MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold sim60 MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold MxFold2 REDfold UFold sincFold sim70 MxFold2 REDfold UFold sincFold MxFold2 REDfold

In [30]:
# add classical methods to the summary
for method in classical_methods:
    for similarity in similarities:
      for fold in folds:
        partition_set = splits_sfold[(splits_sfold["fold_number"] == fold) & (splits_sfold["similarity"] == similarity) & (splits_sfold["partition"] == "test")].set_index('id')
        res = classical_summary.loc[partition_set.index]
        res = res[res["method"] == method]
        res["similarity"] = similarity
        res["fold"] = fold
        summary_sfold = pd.concat([summary_sfold, res])
summary_sfold.method.unique()

array(['MxFold2', 'REDfold', 'UFold', 'sincFold', 'RNAfold',
       'RNAstructure', 'LinearFoldV', 'LinearPartitionV', 'ProbKnot',
       'IPKnot'], dtype=object)

In [31]:
AUCs = {}
wAUCs = {}
for method in summary_sfold.method.unique():
  wAUC = 0
  AUC = 0
  wacc = 0
  for similarity in similarities:
      f1mean = summary_sfold[(summary_sfold["method"] == method) & (summary_sfold["similarity"] == similarity)].f1.mean()
      AUC += f1mean
      w = 1-float(similarity[3:])/100
      wAUC += w * f1mean
      wacc += w
      #print(f"{similarity}: {f1mean}, {w}")

  AUC /= len(similarities)
  wAUC /= wacc

  AUCs[method] = float(AUC)
  wAUCs[method] = float(wAUC)

AUCsim = pd.DataFrame([AUCs, wAUCs], index=["AUC", "wAUC"]).T
#AUCsim

In [32]:
pd.to_pickle({"summary_sfold": summary_sfold, "AUCsim": AUCsim}, DATA_PATH + "results/summary_sfold.pkl")